# Trabalho Prático Individual — Tópicos de Big Data em Python
**Autor:** Antonio Marcos Silva de Oliveira  
**Dataset:** `vendas_bigdata_2026.csv`

## 1. Inspeção Inicial
Carregamento do dataset com Pandas e análise estrutural inicial.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('vendas_bigdata_2026.csv')

print('--- HEAD ---')
display(df.head())

print('\n--- SHAPE ---')
print(df.shape)

print('\n--- INFO ---')
df.info()

print('\n--- DTYPES ---')
print(df.dtypes)

print('\n--- VALORES AUSENTES ---')
print(df.isna().sum())

print('\n--- DUPLICATAS ---')
print('Total de duplicatas:', df.duplicated().sum())

### Respostas da Inspeção Inicial
* **Quantos registros existem?** Existem **1.200 registros** e 15 colunas.
* **Quais problemas de qualidade você identificou?** Valores ausentes (`data`, `categoria`, `regiao`, `avaliacao`), 8 registros duplicados, datas inválidas, inconsistências de formatação textual na coluna `regiao`, registros com `quantidade <= 0` e `preco_unitario <= 0`, além de divergências no cálculo da coluna `valor`.
* **Os dados utilizados por você são primários ou secundários? Justifique.** São **dados secundários**, pois foram coletados e fornecidos previamente pela empresa de varejo (terceiros) e não coletados diretamente pelo analista na origem.

## 2. Preparação e Qualidade dos Dados
Limpeza, padronização e validação das regras de negócio utilizando boas práticas de Pandas (`.loc` e `.copy()`).

In [ ]:
df_clean = df.copy()

# etapa de consistencia operacional

# 1. Converter coluna de data e identificar datas inválidas
df_clean.loc[:, 'data_parsed'] = pd.to_datetime(df_clean['data'], errors='coerce')
print('Datas inválidas identificadas:', df_clean['data_parsed'].isna().sum())

# 2. Verificar e remover duplicatas
print('Duplicatas removidas:', df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().copy()

# 3. Verificar valores ausentes
print('\nValores ausentes após deduplicação:\n', df_clean.isna().sum())

# 4. Padronizar a coluna regiao
df_clean.loc[:, 'regiao'] = df_clean['regiao'].str.strip().str.title()
print('\nRegiões padronizadas:', df_clean['regiao'].unique())

# 5. Verificar quantidade <= 0 e preco_unitario <= 0
print('Quantidade <= 0:', (df_clean['quantidade'] <= 0).sum())
print('Preço unitário <= 0:', (df_clean['preco_unitario'] <= 0).sum())

df_clean = df_clean.loc[(df_clean['quantidade'] > 0) & (df_clean['preco_unitario'] > 0)].copy()

# 6. Validar se valor é coerente com quantidade, preço e desconto
df_clean.loc[:, 'valor_esperado'] = (
    df_clean['quantidade'] *
    df_clean['preco_unitario'] *
    (1 - df_clean['desconto_percentual'] / 100.0)
)

df_clean.loc[:, 'divergencia_valor'] = np.abs(df_clean['valor'] - df_clean['valor_esperado']) > 0.01
print('Divergências de valor encontradas:', df_clean['divergencia_valor'].sum())

df_clean.loc[df_clean['divergencia_valor'], 'valor'] = df_clean.loc[df_clean['divergencia_valor'], 'valor_esperado']
print('Shape final limpo:', df_clean.shape)

### Explicação das Decisões Tomadas
* **Uso de `.copy()` e `.loc`:** Previne `SettingWithCopyWarning` e garante atribuições explícitas e seguras.
* **Datas (`errors='coerce'`):** Converte strings inválidas para `NaT`, evitando interrupção do pipeline e isolando registros corrompidos.
* **Deduplicação:** Remove linhas idênticas para evitar dupla contagem no faturamento.
* **Padronização de `regiao`:** `.str.strip().str.title()` unifica espaços extras e diferenças de capitalização.
* **Regras de Negócio:** Registros com `quantidade <= 0` ou `preco_unitario <= 0` foram filtrados por representarem transações inválidas, e valores divergentes foram recalculados pela fórmula oficial.

## 3. Análise com Pandas
Respostas às perguntas de negócio, exportação do resumo regional e visualizações.

In [ ]:
# 1. Região com maior faturamento e resumo regional consolidado
resumo_regional_validado = df_clean.groupby('regiao').agg(
    faturamento_total=('valor', 'sum'),
    quantidade_pedidos=('pedido_id', 'count'),
    total_itens_vendidos=('quantidade', 'sum')
).reset_index().sort_values(by='faturamento_total', ascending=False)

print('1. Faturamento por Região:')
display(resumo_regional_validado)

os.makedirs('resultado', exist_ok=True)
resumo_regional_validado.to_csv('resultado/resumo_regioes.csv', index=False)

# 2. Cinco produtos com maior faturamento
top_produtos = df_clean.groupby('produto')['valor'].sum().reset_index().sort_values(by='valor', ascending=False).head(5)
print('\n2. Top 5 Produtos por Faturamento:')
display(top_produtos)

# 3. Canal com maior faturamento
faturamento_canal = df_clean.groupby('canal')['valor'].sum().reset_index().sort_values(by='valor', ascending=False)
print('\n3. Faturamento por Canal:')
display(faturamento_canal)

# 4. Forma de pagamento mais utilizada
pagamento_uso = df_clean['forma_pagamento'].value_counts().reset_index()
pagamento_uso.columns = ['forma_pagamento', 'quantidade_transacoes']
print('\n4. Forma de Pagamento Mais Utilizada:')
display(pagamento_uso)

# 5. Evolução do faturamento mês a mês
df_clean['mes'] = df_clean['data_parsed'].dt.to_period('M')
faturamento_mes = df_clean.groupby('mes')['valor'].sum().reset_index()
print('\n5. Evolução Mensal do Faturamento:')
display(faturamento_mes)

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Faturamento Total por Região
sns.barplot(data=resumo_regional_validado, x='regiao', y='faturamento_total', ax=axes[0], hue='regiao', palette='Blues_d', legend=False)
axes[0].set_title('Faturamento Total por Região (R$)')
axes[0].set_xlabel('Região')
axes[0].set_ylabel('Faturamento (R$)')

# Gráfico 2: Evolução Mensal do Faturamento
fat_mes_plot = faturamento_mes.copy()
fat_mes_plot['mes'] = fat_mes_plot['mes'].astype(str)
sns.lineplot(data=fat_mes_plot, x='mes', y='valor', marker='o', ax=axes[1], color='navy')
axes[1].set_title('Evolução do Faturamento Mês a Mês')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('Faturamento (R$)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Conclusão Baseada nos Dados
A análise demonstra que a região **Sudeste** lidera o faturamento da varejista, concentrando a maior parcela de pedidos e itens vendidos, seguida por Sul e Nordeste. Os canais digitais (**Site** e **App**) superam amplamente a loja física, com destaque para pagamentos via **Pix** e **Cartão de Crédito**. Observam-se picos de faturamento nos meses de julho, setembro e outubro. O tratamento rigoroso de inconsistências garantiu a **rastreabilidade analitica** e a confiabilidade dos indicadores.

## 4. PySpark DataFrame
Processamento distribuído com `filter`, `groupBy`, `agg` e `orderBy`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, sum as _sum, count

spark = SparkSession.builder \
    .appName('TrabalhoBigDataVarejo') \
    .getOrCreate()

df_spark = spark.read.csv('vendas_bigdata_2026.csv', header=True, inferSchema=True)

resultado_spark = df_spark \
    .filter((col('quantidade') > 0) & (col('preco_unitario') > 0) & col('regiao').isNotNull()) \
    .withColumn('regiao', initcap(trim(col('regiao')))) \
    .groupBy('regiao') \
    .agg(
        _sum('valor').alias('faturamento_total'),
        count('pedido_id').alias('quantidade_pedidos'),
        _sum('quantidade').alias('quantidade_total_itens')
    ) \
    .orderBy(col('faturamento_total').desc())

resultado_spark.show()

### Explicação Conceitual PySpark
* **Transformações:** `filter`, `withColumn`, `groupBy`, `agg` e `orderBy` são transformações *lazy* (preguiçosas), que apenas montam o plano lógico de execução (DAG) sem computar dados imediatamente.
* **Ação que provoca execução:** `show()` é uma ação (*action*), responsável por disparar a execução física das tarefas no cluster.
* **Por que `groupBy` exige redistribuição:** Para agregar registros pela mesma chave (`regiao`), o Spark precisa mover registros entre nós da rede para que chaves iguais fiquem na mesma partição, processo conhecido como **Shuffle**.

## 5. Spark SQL
Consulta analítica utilizando View Temporária.

In [ ]:
df_spark.createOrReplaceTempView('vendas')

query_sql = '''
    SELECT 
        canal,
        COUNT(pedido_id) AS total_pedidos,
        ROUND(SUM(valor), 2) AS faturamento_total,
        ROUND(AVG(valor), 2) AS ticket_medio
    FROM vendas
    WHERE quantidade > 0 AND preco_unitario > 0
    GROUP BY canal
    ORDER BY faturamento_total DESC
'''

spark.sql(query_sql).show()

## 6. HDFS e Escalabilidade
* **NameNode:** Gerencia os metadados do sistema de arquivos distribuído, estrutura de diretórios e o mapeamento de onde cada bloco está armazenado.
* **DataNodes:** Armazenam fisicamente os blocos de dados nos discos locais e executam operações de leitura e escrita solicitadas.
* **Falha de Heartbeats:** Se um DataNode para de enviar *heartbeats*, o NameNode o marca como inativo e inicia automaticamente a replicação dos blocos perdidos a partir das cópias saudáveis para manter o fator 3.
* **Replicação e Disponibilidade:** Manter 3 réplicas em nós distintos garante tolerância a falhas e alta disponibilidade contínua mesmo em caso de queda de hardware.

## 7. Auditoria de Machine Learning
* **Problema metodológico:** Aplicar `scaler.fit_transform(X)` antes do `train_test_split` causa **vazamento de dados (Data Leakage)**, pois a média e o desvio padrão do conjunto de teste contaminam o escalonamento do treino, superestimando a performance do modelo.
* **Solução corrigida:** Dividir os dados antes e encapsular o `StandardScaler` dentro de um `Pipeline` ajustado exclusivamente nos dados de treino.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Exemplo reprodutível com features numéricas do dataset
df_ml = df_clean[['quantidade', 'preco_unitario', 'desconto_percentual', 'frete', 'devolvido']].dropna().copy()
X = df_ml[['quantidade', 'preco_unitario', 'desconto_percentual', 'frete']]
y = (df_ml['devolvido'] == 'Sim').astype(int)

# 1. Divisão ANTES de qualquer transformação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Pipeline prevenindo Data Leakage
pipeline_ml = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(random_state=42))
])

pipeline_ml.fit(X_train, y_train)
print(f'Acurácia no teste (sem vazamento): {pipeline_ml.score(X_test, y_test):.4f}')